In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Kaggle". Running its setup...
Updating the package lists...
Installing nvidia-cuda-toolkit, this may take a few minutes...
Setup failed for detected platform "Kaggle". Set the "NVCC4JUPYTER_NO_SETUP" environment variable to disable running the setup on load. Please report the following error to https://github.com/andreinechaev/nvcc4jupyter/issues: following error message:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/nvcc4jupyter/setup_env.py", line 60, in setup_environment
    kaggle_setup()
  File "/usr/local/lib/python3.12/dist-packages/nvcc4jupyter/setup_env.py", line 35, in kaggle_setup
    os.symlink(KAGGLE_GCC_8_PATH, gcc_symlink_path)
FileExistsError: [Errno 17] File exists: '/usr/bin/gcc-8' -> '/usr/bin/priority/gcc'

Source files will be saved in "/tmp/tmpdxbi4hs4".


### Vector Addition

- `N` blocks w/ `1` thread: `blockIdx.x`

In [2]:
%%cuda

#include <iostream>

#define N 32

__global__ void add(int *a, int *b, int *c){
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

int main(void){

    int size = N * sizeof(int);
    int *a, *b, *c; // Host copies of a, b, c
    int *d_a, *d_b, *d_c; // Device copies of a, b, c

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c
    a = (int*)malloc(size);
    b = (int*)malloc(size);
    c = (int*)malloc(size);

    // Setup input values
    for (int i=0; i<N; i++){
        a[i] = 1;
        b[i] = 3;
    }

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU
    add<<<N, 1>>>(d_a, d_b, d_c);

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i=0; i<N; i++){
        printf("%d ", c[i]);
    }

    // Cleanup
    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 


- `1` block w/ `N` threads: `threadIdx.x`

In [3]:
%%cuda

#include <iostream>

#define N 32

__global__ void add(int *a, int *b, int *c){
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

int main(void){

    int size = N * sizeof(int);
    int *a, *b, *c; // Host copies of a, b, c
    int *d_a, *d_b, *d_c; // Device copies of a, b, c

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c
    a = (int*)malloc(size);
    b = (int*)malloc(size);
    c = (int*)malloc(size);

    // Setup input values
    for (int i=0; i<N; i++){
        a[i] = 1;
        b[i] = 3;
    }

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU
    add<<<1, N>>>(d_a, d_b, d_c);

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i=0; i<N; i++){
        printf("%d ", c[i]);
    }

    // Cleanup
    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 


Using `threads/block`: `idx = threadIdx.x + blockIdx.x * blockDim.x`

In [4]:
%%cuda

#include <iostream>

#define N 32
#define THREADS_PER_BLOCK 4

__global__ void add(int *a, int *b, int *c){
    int index = threadIdx.x + blockIdx.x * blockDim.x;
    c[index] = a[index] + b[index];
}

int main(void){

    int size = N * sizeof(int);
    int *a, *b, *c; // Host copies of a, b, c
    int *d_a, *d_b, *d_c; // Device copies of a, b, c

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c
    a = (int*)malloc(size);
    b = (int*)malloc(size);
    c = (int*)malloc(size);

    // Setup input values
    for (int i=0; i<N; i++){
        a[i] = 1;
        b[i] = 3;
    }

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU
    add<<<N / THREADS_PER_BLOCK, THREADS_PER_BLOCK>>>(d_a, d_b, d_c);

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i=0; i<N; i++){
        printf("%d ", c[i]);
    }

    // Cleanup
    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 


Arbitrary vector sizes

In [5]:
%%cuda

#include <iostream>

#define N 50
#define THREADS_PER_BLOCK 4

__global__ void add(int *a, int *b, int *c, int n){
    int index = threadIdx.x + blockIdx.x * blockDim.x;
    if (index < n){
        c[index] = a[index] + b[index];
    }
}

int main(void){

    int size = N * sizeof(int);
    int *a, *b, *c; // Host copies of a, b, c
    int *d_a, *d_b, *d_c; // Device copies of a, b, c

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c
    a = (int*)malloc(size);
    b = (int*)malloc(size);
    c = (int*)malloc(size);

    // Setup input values
    for (int i=0; i<N; i++){
        a[i] = 1;
        b[i] = 3;
    }

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU
    add<<<(N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i=0; i<N; i++){
        printf("%d ", c[i]);
    }

    // Cleanup
    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
